# 🏆 Notebook 07: BERTized Agentic Context Engineering (ACE) Pipeline
### High-Precision Lifestyle Factor (LSF) Extraction in a Post-BERT Proposer Setting
#### Target Dataset: Combined 200Abstracts Train (120) + Val (40) = 160 Abstract Pairs

---

## 📌 Architectural Overview

This notebook implements the **BERTized Agentic Context Engineering (ACE)** pipeline. Rather than forcing generative LLMs to scan full raw abstracts from scratch (which leads to severe hallucination and boundary instability), the system operates in a **two-phase decoupled architecture**:

```
                                  ┌─────────────────────────────────────────────────────────────┐
                                  │             PHASE 1: SPANNER NEURAL PROPOSER                │
  Raw Abstract Text ─────────────►│  • Proposes candidate spans (start_char, end_char)          │
                                  │  • Computes top-1, top-2, uncertainty, novelty, and margin  │
                                  └──────────────────────────────┬──────────────────────────────┘
                                                                 │ Candidate Proposals + BERT Evidence
                                                                 ▼
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                           PHASE 2: 3-AGENT COLLABORATIVE ACE CURATION LOOP                                     │
│                                                                                                                │
│   ┌───────────────────────────┐      ┌───────────────────────────┐      ┌───────────────────────────┐          │
│   │   1. ACE GENERATOR (v2)   │      │   2. ACE REFLECTOR (v2)   │      │    3. ACE CURATOR (v2)    │          │
│   │  • Reads BERT evidence    │      │  • Evaluates in-scope     │      │  • Screens against BASE   │          │
│   │  • Classifies candidate   │─────►│    BERT candidate errors  │─────►│    RULES & 4 Genericity   │          │
│   │    spans (9 LSF, Out, O)  │      │  • Checks evidence anchor │      │    Tests                  │          │
│   │  • Cites dynamicGuideline │      │  • Formulates key insight │      │  • Formulates ADD/MODIFY  │          │
│   └───────────────────────────┘      └───────────────────────────┘      └─────────────┬─────────────┘          │
│                 ▲                                                                     │                        │
│                 │                                                                     ▼                        │
│                 │                    ┌───────────────────────────────────┐     Guideline Operations    │
│                 └────────────────────│    DYNAMIC GUIDELINES MANAGER     │◄────────────────────────────┘       │
│                  Current Playbook    │  • Persistent vector store (Chroma│                                     │
│                                      │  • Auto pruning & helpful/harmful │                                     │
│                                      └───────────────────────────────────┘                                     │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### 1. The 3 Specialized ACE Agents (v2):
1. **Generator (`ACE_Generator_v2.txt`)**: Operates exclusively on candidate spans proposed by SpanNER. Weighs `predicted_label` vs `second_best_label` based on `margin`, `uncertainty`, and `novelty_score`. Applies Base Rules and dynamic guidelines. Max tokens: **16k**.
2. **Reflector (`ACE_Reflector_v2.txt`)**: Diagnostic analyst covering **ONLY in-scope BERT-proposed candidate spans**. Evaluates `Comparison_Label` (`TP`, `FP`, `FN`, `CLSS_ERR`), detects evidence anchoring, identifies root cause, and derives generalizable key insights.
3. **Curator (`ACE_Curator_v2.txt`)**: Senior policy architect acting as the last line of defense. Screens insights against Base Rules and 4 strict genericity tests (Pattern, Necessity, Restatement, Stability) before generating structured `ADD` or `MODIFY` operations.

### 2. Dual-View Standardized Evaluation:
* **View 1 (Candidate-Level Arbitration)**: Evaluates decision accuracy on candidate mentions proposed by SpanNER (`TP / (TP + FN)` across candidate mentions).
* **View 2 (Global End-to-End Dataset Recovery)**: Evaluates total recovery against gold entities in the dataset across standard academic formulations (**Strict CoNLL**, **Partial MUC-7**, and **Candidate Mention**).


In [1]:
# ==============================================================================
# [CELL 1]: Environment Setup, Paths & Imports
# ==============================================================================
import os
import sys
import json
import time
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dotenv import load_dotenv

# Robustly resolve PROJECT_ROOT whether running from /notebooks or project root
if os.path.exists("data") and os.path.exists("src"):
    PROJECT_ROOT = os.path.abspath(".")
elif os.path.exists("../data") and os.path.exists("../src"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Explicitly load .env configuration
dotenv_path = os.path.join(PROJECT_ROOT, ".env")
load_dotenv(dotenv_path, override=True)

import torch
import numpy as np

# Core Pipeline & Architecture Imports
from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline
from src.architectures.agentic_guidelines.manager import DynamicGuidebookManager, DynamicGuidelinesManager
from src.architectures.agentic_guidelines.bertized_ace import (
    BertizedGenerator,
    BertizedReflector,
    BertizedCurator,
    BertizedACEPipeline
)
from src.data.cached_loader import load_cached_spanner_candidates, get_cache_split_key
from src.common.audit_logger import (
    classify_10_outcome_str,
    classify_8_outcome_str,
    match_gt_label,
    create_run_directory
)
from src.common.label_mapping import canonicalize_label
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box
)

print(f"✅ Environment initialized. Project Root: {PROJECT_ROOT}")
print(f"⚡ Device: {'CUDA' if torch.cuda.is_available() else ('MPS' if torch.backends.mps.is_available() else 'CPU')}")


/Users/hmd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Environment initialized. Project Root: /Users/hmd/Documents/Workspace/Unified_Test
⚡ Device: MPS


In [2]:
# ==============================================================================
# [CELL 2]: Experiment Configuration (400Abstracts Train: 240, Val: 80, B=5, Multi-Threaded)
# ==============================================================================
SEED = int(os.getenv("RANDOM_SEED", "42"))
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Target Datasets:
# Training split (240 abstracts): data/new/400Abstracts/no_disease/train
# Testing/Validation split (80 abstracts): data/new/400Abstracts/no_disease/val
TRAIN_DATA_DIR = os.getenv("TRAIN_DATA_DIR", os.path.join(PROJECT_ROOT, "data/new/400Abstracts/no_disease/train"))
if not os.path.exists(TRAIN_DATA_DIR):
    TRAIN_DATA_DIR = os.path.join(PROJECT_ROOT, "data/new/400Abstracts/train")

VAL_DATA_DIR = os.getenv("VAL_DATA_DIR", os.path.join(PROJECT_ROOT, "data/new/400Abstracts/no_disease/val"))
if not os.path.exists(VAL_DATA_DIR):
    VAL_DATA_DIR = os.path.join(PROJECT_ROOT, "data/new/400Abstracts/val")

DATASET_DIRS = [TRAIN_DATA_DIR, VAL_DATA_DIR]

# 2. Output & Guideline Storage
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output/bertized_ace")
DYNAMIC_GUIDEBOOK_PATH = os.path.join(PROJECT_ROOT, "data/guidelines/dynamicGuidebook.json")
DYNAMIC_GUIDELINES_PATH = DYNAMIC_GUIDEBOOK_PATH
CHROMA_DIR = os.path.join(PROJECT_ROOT, "data/guidelines/chroma_db")

# 2b. ACE Curation Policy Mode:
# "per_span": Evaluates each misclassified span individually (up to 1 rule per erroneous span)
# "per_abstract": Evaluates combined abstract reflection (at most 1 rule per abstract)
CURATOR_MODE = os.getenv("CURATOR_MODE", "per_span")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

# 3. Model & LLM Provider Configuration (.env driven: 9Router / gemini-cli, 16k Max Tokens)
MODEL_NAME = os.getenv("LLM_MODEL", "gemini-cli")
BACKEND = os.getenv("LLM_BACKEND", "api")
PROVIDER = os.getenv("LLM_PROVIDER", "9router")
API_BASE_URL = os.getenv("LLM_API_BASE_URL", os.getenv("API_Base_URL", "http://localhost:20128/v1"))
MAX_TOKENS = 16384  # 16k Max Tokens for deep reflection & guideline reasoning

# 4. SpanNER Neural Proposer & Batching Configuration
SPANNER_CHECKPOINT = os.path.join(PROJECT_ROOT, "models/SpanNER_LSF/best_spanner_160train.pt")
USE_NEW_LABELS = True
BATCH_SIZE = 5      # Batch size: 5 abstracts per batch
MAX_WORKERS = 5     # 5 parallel threads per batch
USE_NMS = False

# 5. Cached SpanNER Output Configuration (High-Speed & Exact Reproducibility)
USE_CACHED_SPANNER = os.getenv("USE_CACHED_SPANNER", "true").lower() in ("true", "1", "yes")
CACHED_SPANNER_DIR = os.path.join(PROJECT_ROOT, "data/cached/spanner_160_10_class")


# 6. Selective Gating & Escalation Configuration (SpanNER Candidate Filtering)
# Only ambiguous / out-of-distribution candidate spans are escalated to the Generator LLM.
# Confident spans retain SpanNER predictions directly (0 LLM cost).
ENABLE_GATING_TRAIN = os.getenv("ENABLE_GATING_TRAIN", "true").lower() in ("true", "1", "yes")
ENABLE_GATING_TEST = os.getenv("ENABLE_GATING_TEST", "true").lower() in ("true", "1", "yes")
GATING_MODE = os.getenv("GATING_MODE", "prob_or")  # "prob_or" (optimal ablation: 1 - (1-nov)*(1-unc))
UNRELIABILITY_THRESHOLD = float(os.getenv("UNRELIABILITY_THRESHOLD", "0.35"))
NOVELTY_THRESHOLD = float(os.getenv("NOVELTY_THRESHOLD", "0.55"))
W_UNCERTAINTY = float(os.getenv("W_UNCERTAINTY", "0.70"))
W_NOVELTY = float(os.getenv("W_NOVELTY", "0.30"))
INCLUDE_CONFIDENT_AS_READONLY = os.getenv("INCLUDE_CONFIDENT_AS_READONLY", "false").lower() in ("true", "1", "yes")

# 7. Margin Safeguard Veto (Protects High-Margin SpanNER Mentions)
ENABLE_MARGIN_SAFEGUARD_VETO = os.getenv("ENABLE_MARGIN_SAFEGUARD_VETO", "true").lower() in ("true", "1", "yes")
ARBITRATION_MARGIN_THRESHOLD = float(os.getenv("MARGIN_THRESHOLD", "0.50"))
P_O_THRESHOLD = float(os.getenv("P_O_THRESHOLD", "0.20"))
print("=" * 80)
print("📋 BERTIZED ACE MASTER CONFIGURATION")
print("=" * 80)
print(f"  • Training Dataset Dir   : {TRAIN_DATA_DIR} (240 abstracts)")
print(f"  • Validation Dataset Dir : {VAL_DATA_DIR} (80 abstracts)")
print(f"  • Batch Size (B) / Workers: {BATCH_SIZE} abstracts per batch ({MAX_WORKERS} parallel threads)")
print(f"  • SpanNER Model Checkpoint: {SPANNER_CHECKPOINT}")
print(f"  • LLM Provider / Gateway : {PROVIDER.upper()} -> {API_BASE_URL}")
print(f"  • Active Model Identifier : {MODEL_NAME}")
print(f"  • Max Tokens for ACE Pipeline: {MAX_TOKENS} (16k)")
print(f"  • Curator Operating Mode : {CURATOR_MODE.upper()} ({'Multi-operation per span' if CURATOR_MODE == 'per_span' else 'At most 1 operation per abstract'})")
print(f"  • Dynamic Guidebook JSON : {DYNAMIC_GUIDEBOOK_PATH}")
print(f"  • ChromaDB Vector Store  : {CHROMA_DIR}")
print(f"  • Target Label Schema    : {'NEW (Relabeled)' if USE_NEW_LABELS else 'LEGACY'}")
print("=" * 80)


📋 BERTIZED ACE MASTER CONFIGURATION
  • Training Dataset Dir   : /Users/hmd/Documents/Workspace/Unified_Test/data/new/400Abstracts/no_disease/train (240 abstracts)
  • Validation Dataset Dir : /Users/hmd/Documents/Workspace/Unified_Test/data/new/400Abstracts/no_disease/val (80 abstracts)
  • Batch Size (B) / Workers: 5 abstracts per batch (5 parallel threads)
  • SpanNER Model Checkpoint: /Users/hmd/Documents/Workspace/Unified_Test/models/SpanNER_LSF/best_spanner_160train.pt
  • LLM Provider / Gateway : 9ROUTER -> http://localhost:20128/v1
  • Active Model Identifier : gemini-cli
  • Max Tokens for ACE Pipeline: 16384 (16k)
  • Curator Operating Mode : PER_SPAN (Multi-operation per span)
  • Dynamic Guidebook JSON : /Users/hmd/Documents/Workspace/Unified_Test/data/guidelines/dynamicGuidebook.json
  • ChromaDB Vector Store  : /Users/hmd/Documents/Workspace/Unified_Test/data/guidelines/chroma_db
  • Target Label Schema    : NEW (Relabeled)


In [3]:
# ==============================================================================
# [CELL 3]: Load 400Abstracts Dataset Pairs (.txt + .ann pairs)
# Robust parser handles fragmented/discontinuous BRAT spans with semicolons
# ==============================================================================
def load_abstract_pairs_from_dir(d_dir):
    pairs = []
    if not os.path.exists(d_dir):
        print(f"⚠️ Directory not found: {d_dir}")
        return pairs
    txt_files = sorted([f for f in os.listdir(d_dir) if f.endswith(".txt")])
    for txt_file in txt_files:
        base = os.path.splitext(txt_file)[0]
        txt_path = os.path.join(d_dir, f"{base}.txt")
        ann_path = os.path.join(d_dir, f"{base}.ann")
        
        gt_ents = []
        if os.path.exists(ann_path):
            with open(ann_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line or not line.startswith("T"):
                        continue
                    parts = line.split("	")
                    if len(parts) < 3:
                        continue
                    type_offsets = parts[1]
                    ent_text = parts[2]
                    type_parts = type_offsets.split()
                    lbl = type_parts[0]
                    # Robustly parse offsets, supporting fragmented spans with semicolons (e.g. '811 820;837 855')
                    raw_offsets = type_offsets[len(lbl):].strip().replace(";", " ").split()
                    if len(raw_offsets) >= 2:
                        st_c = int(raw_offsets[0])
                        end_c = int(raw_offsets[-1])
                        gt_ents.append({
                            "label": canonicalize_label(lbl, use_new_labels=USE_NEW_LABELS),
                            "start_char": st_c,
                            "end_char": end_c,
                            "text": ent_text
                        })
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read()
        pairs.append((txt_path, ann_path, text, gt_ents, base))
    return pairs

train_files = load_abstract_pairs_from_dir(TRAIN_DATA_DIR)
test_files = load_abstract_pairs_from_dir(VAL_DATA_DIR)

print(f"📦 Successfully loaded 400Abstracts dataset pairs:")
print(f"  • Training / Curation Split (400/train): {len(train_files)} abstracts")
print(f"  • Validation / Test Split   (400/val)  : {len(test_files)} abstracts")


📦 Successfully loaded 400Abstracts dataset pairs:
  • Training / Curation Split (400/train): 240 abstracts
  • Validation / Test Split   (400/val)  : 80 abstracts


In [4]:
# ==============================================================================
# [CELL 4]: Phase 1 - Fast Neural Batch Candidate Proposer (SpanNER + Qwen-8B Novelty)
# ==============================================================================
if USE_CACHED_SPANNER:
    print("=" * 80)
    print("🚀 Loading Pre-Computed SpanNER Candidate Spans directly from Cache...")
    print(f"  • Source Directory: {CACHED_SPANNER_DIR}")
    print("=" * 80)
    t0 = time.time()

    train_candidates_cache = load_cached_spanner_candidates(
        split_key="400/train",
        cache_dir=CACHED_SPANNER_DIR,
        doc_ids=[item[4] for item in train_files]
    )
    test_candidates_cache = load_cached_spanner_candidates(
        split_key="400/val",
        cache_dir=CACHED_SPANNER_DIR,
        doc_ids=[item[4] for item in test_files]
    )

    # Attach ground truth entities from loaded files for downstream auditing & evaluation
    for item in train_files:
        doc_id, gt_ents = item[4], item[3]
        if doc_id in train_candidates_cache:
            train_candidates_cache[doc_id]["gt_ents"] = gt_ents
    for item in test_files:
        doc_id, gt_ents = item[4], item[3]
        if doc_id in test_candidates_cache:
            test_candidates_cache[doc_id]["gt_ents"] = gt_ents

    total_train_spans = sum(len(d["candidates"]) for d in train_candidates_cache.values())
    total_test_spans = sum(len(d["candidates"]) for d in test_candidates_cache.values())
    elapsed = time.time() - t0
    print(f"⚡ [CACHE HIT] Successfully loaded candidates in {elapsed:.3f}s!")
    print(f"  • Train Candidates (400/train): {total_train_spans} spans across {len(train_candidates_cache)} abstracts")
    print(f"  • Test Candidates  (400/val)  : {total_test_spans} spans across {len(test_candidates_cache)} abstracts")
    spanner_pipeline = None
else:
    print("=" * 80)
    print("🚀 Initializing Live SpanNER Neural Proposer Pipeline...")
    print("=" * 80)

    spanner_pipeline = LinkNERPipeline(
        spanner_checkpoint_path=SPANNER_CHECKPOINT,
        use_new_labels=USE_NEW_LABELS,
        use_nms=USE_NMS
    )

    def extract_candidates_for_split(files_list, split_name="Split"):
        t0 = time.time()
        texts = [item[2] for item in files_list]
        print(f"⚡ Running SpanNER neural forward passes on {len(texts)} {split_name} abstracts (B={BATCH_SIZE})...")
        batch_raw_preds = spanner_pipeline.predict_batch(
            texts=texts,
            batch_size=BATCH_SIZE,
            uncertainty_threshold=1.01,
            use_nms=USE_NMS,
            verbose=False
        )
        extracted_records = {}
        total_spans = 0
        for idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(files_list):
            raw_cands = batch_raw_preds[idx]
            processed_cands = []
            for c in raw_cands:
                s_text = c["span_text"]
                st_c = c["start_char"]
                end_c = c["end_char"]
                p_lbl = canonicalize_label(c.get("predicted_label", c.get("spanner_label", "O")), use_new_labels=USE_NEW_LABELS)
                s_lbl = canonicalize_label(c.get("second_best_label", "O"), use_new_labels=USE_NEW_LABELS)
                unc = float(c.get("uncertainty", 0.0))
                nov = float(c.get("novelty", 0.0))
                margin = float(c.get("margin", 0.0))
                gt_lbl = match_gt_label(s_text, st_c, end_c, gt_ents)
                cand_dict = {
                    "span_text": s_text,
                    "entity": s_text,
                    "start_char": st_c,
                    "end_char": end_c,
                    "predicted_label": p_lbl,
                    "spanner_label": p_lbl,
                    "second_best_label": s_lbl,
                    "uncertainty": unc,
                    "novelty_score": nov,
                    "margin": margin,
                    "prob": float(c.get("prob", 0.0)),
                    "gt_label": gt_lbl
                }
                processed_cands.append(cand_dict)
                total_spans += 1
            extracted_records[doc_id] = {
                "text": text,
                "candidates": processed_cands,
                "gt_ents": gt_ents
            }
        elapsed = time.time() - t0
        print(f"✅ Extracted {total_spans} candidate spans across {len(files_list)} abstracts in {elapsed:.2f}s.")
        return extracted_records

    train_candidates_cache = extract_candidates_for_split(train_files, split_name="Train (240)")
    test_candidates_cache = extract_candidates_for_split(test_files, split_name="Val (80)")


🚀 Loading Pre-Computed SpanNER Candidate Spans directly from Cache...
  • Source Directory: /Users/hmd/Documents/Workspace/Unified_Test/data/cached/spanner_160_10_class
⚡ [CACHE HIT] Successfully loaded candidates in 0.044s!
  • Train Candidates (400/train): 3713 spans across 240 abstracts
  • Test Candidates  (400/val)  : 1492 spans across 80 abstracts


In [5]:
# ==============================================================================
# [CELL 5]: Initialize BERTized ACE Pipeline & Clean-Slate Dynamic Guidelines Store
# ==============================================================================
import importlib
import src.architectures.agentic_guidelines.manager as mgr_mod
import src.architectures.agentic_guidelines.bertized_ace as ace_mod
importlib.reload(mgr_mod)
importlib.reload(ace_mod)
from src.architectures.agentic_guidelines.bertized_ace import BertizedACEPipeline

print("=" * 80)
print("🏛️ Initializing DynamicGuidelinesManager (ChromaDB + JSON) & BERTized ACE Pipeline...")
print("=" * 80)

ace_pipeline = BertizedACEPipeline(
    model_name=MODEL_NAME,
    backend=BACKEND,
    dynamicGuidelines_path=DYNAMIC_GUIDELINES_PATH,
    chroma_persist_directory=CHROMA_DIR,
    spanner_pipeline=spanner_pipeline,
    max_tokens=MAX_TOKENS,
    curator_mode=CURATOR_MODE,
    enable_gating_train=ENABLE_GATING_TRAIN,
    enable_gating_test=ENABLE_GATING_TEST,
    gating_mode=GATING_MODE,
    unreliability_threshold=UNRELIABILITY_THRESHOLD,
    novelty_threshold=NOVELTY_THRESHOLD,
    w_uncertainty=W_UNCERTAINTY,
    w_novelty=W_NOVELTY,
    include_confident_as_readonly=INCLUDE_CONFIDENT_AS_READONLY,
    enable_margin_safeguard_veto=ENABLE_MARGIN_SAFEGUARD_VETO,
    margin_threshold=ARBITRATION_MARGIN_THRESHOLD,
    p_o_threshold=P_O_THRESHOLD,
)

# Check for existing training checkpoint before clean-slate reset
TRAIN_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "train_checkpoint.json")
has_active_checkpoint = os.path.exists(TRAIN_CHECKPOINT_PATH)
FORCE_RESET = os.getenv("RESET_GUIDELINES", "").lower() == "force"

# Clean-slate initialization per user instruction (automatic archive + clean empty start)
RESET_GUIDELINES_FOR_TRAIN = os.getenv("RESET_GUIDELINES", "true").lower() in ("true", "1", "yes")

if has_active_checkpoint and not FORCE_RESET:
    print("=" * 80)
    print(f"🔄 Active training checkpoint detected: {TRAIN_CHECKPOINT_PATH}")
    print("   Preserving existing dynamic guidelines in store so training can resume seamlessly!")
    print("=" * 80)
elif RESET_GUIDELINES_FOR_TRAIN:
    if hasattr(ace_pipeline.manager, "reset_to_empty"):
        archive_path = ace_pipeline.manager.reset_to_empty()
    else:
        # Bulletproof inline fallback
        from datetime import datetime
        import shutil
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        arch_dir = os.path.join(os.path.dirname(DYNAMIC_GUIDELINES_PATH), "archive")
        os.makedirs(arch_dir, exist_ok=True)
        archive_path = os.path.join(arch_dir, f"dynamicGuidebook_pre_400train_{ts}.json")
        if os.path.exists(DYNAMIC_GUIDELINES_PATH):
            shutil.copy2(DYNAMIC_GUIDELINES_PATH, archive_path)
        ace_pipeline.manager.store.guidebook = {"guidebook": []}
        ace_pipeline.manager.store._guidelines_by_id = {}
        ace_pipeline.manager.store._max_id_num = 0
        ace_pipeline.manager.store.save(backup=False)
        if ace_pipeline.manager.vectors and ace_pipeline.manager.vectors.collection:
            try:
                ace_pipeline.manager.vectors.client.delete_collection("dynamic_guidebook")
                ace_pipeline.manager.vectors.collection = ace_pipeline.manager.vectors.client.create_collection("dynamic_guidebook")
            except Exception:
                pass

    print("🧹 Clean-slate initialized! Archived previous dynamic guidebook to:")
    print(f"   {archive_path}")

initial_guidelines = ace_pipeline.manager.get_dynamicGuidelines()
initial_bullet_count = sum(len(bullets) for bullets in initial_guidelines.values()) if isinstance(initial_guidelines, dict) else len(initial_guidelines)
print(f"📚 Initial Guidelines Loaded: {initial_bullet_count} active bullets.")


🏛️ Initializing DynamicGuidelinesManager (ChromaDB + JSON) & BERTized ACE Pipeline...
Initialized SentenceTransformer on device: cpu
Loaded existing ChromaDB collection: dynamicGuidebook_collection
Loaded dynamicGuidebook from /Users/hmd/Documents/Workspace/Unified_Test/data/guidelines/dynamicGuidebook.json
ChromaDB and dynamicGuidebook are in sync.
🧹 Clean-slate initialized! Archived previous dynamic guidebook to:
   /Users/hmd/Documents/Workspace/Unified_Test/data/guidelines/archive/dynamicGuidebook_pre_400train_20260920_170819.json
📚 Initial Guidelines Loaded: 0 active bullets.


In [6]:
# ==============================================================================
# [CELL 6]: Phase 2 - Training & Iterative Guideline Curation (Multi-Threaded ACE Loop, B=5)
# ==============================================================================
from pickle import FALSE
# Batch execution control:
#   • Set MAX_BATCHES = 2 (or 3, 5, etc.) for quick test runs on updated prompts.
#   • Set MAX_BATCHES = None (or 0) for the full 48 batches (all 240 abstracts).
MAX_BATCHES = 0

# Set True to test modified prompts from a clean slate; False to resume from checkpoint
FORCE_RESTART = FALSE

# ------------------------------------------------------------------------------
# Subset Partitioning
# ------------------------------------------------------------------------------
if MAX_BATCHES and MAX_BATCHES > 0:
    curation_limit = min(MAX_BATCHES * BATCH_SIZE, len(train_files))
    curation_subset = train_files[:curation_limit]
    print(f"🎯 Targeted Test Mode: Limiting run to {MAX_BATCHES} batch(es) ({len(curation_subset)} abstracts).")
else:
    curation_limit = int(os.getenv("CURATION_LIMIT", "0")) or len(train_files)
    curation_subset = train_files[:curation_limit]
    print(f"🌐 Full Production Mode: Running all {len(curation_subset)} abstracts.")

print("=" * 96)
print(f"🚀 Starting BERTized ACE Multi-Threaded Curation across {len(curation_subset)} abstracts (B={BATCH_SIZE}, Workers={MAX_WORKERS}, Mode={CURATOR_MODE})...")
print("=" * 96)

# ------------------------------------------------------------------------------
# Batch-Level Checkpoint & Recovery Mechanism
# ------------------------------------------------------------------------------
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "train_checkpoint.json")
FINAL_LOGS_PATH = os.path.join(OUTPUT_DIR, "train_logs.json")

train_logs = []
last_completed_batch = -1

if not FORCE_RESTART and os.path.exists(CHECKPOINT_PATH):
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            ckpt = json.load(f)
        train_logs = ckpt.get("train_logs", [])
        last_completed_batch = ckpt.get("last_completed_batch", -1)
        print("=" * 80)
        print(f"🔄 Checkpoint detected: {len(train_logs)} abstracts already processed ({last_completed_batch + 1} batches).")
        print(f"   Skipping batches 1..{last_completed_batch + 1}, resuming execution from Batch {last_completed_batch + 2}...")
        print("=" * 80)
    except Exception as e:
        print(f"⚠️ Failed to load checkpoint ({e}). Starting fresh from Batch 1.")
        train_logs = []
        last_completed_batch = -1
elif FORCE_RESTART:
    print("✨ FORCE_RESTART active: Starting clean run from Batch 1 (ignoring previous checkpoints).")

step_start = time.time()
num_batches = (len(curation_subset) + BATCH_SIZE - 1) // BATCH_SIZE

for b_idx in range(num_batches):
    if b_idx <= last_completed_batch:
        continue  # Skip already processed batches

    b_start = b_idx * BATCH_SIZE
    b_end = min(b_start + BATCH_SIZE, len(curation_subset))
    b_items = curation_subset[b_start:b_end]
    
    t_batch_start = time.time()
    print()
    print(f"⚡ [Batch {b_idx + 1}/{num_batches}] Processing abstracts {b_start + 1}..{b_end} in {len(b_items)} parallel threads...")

    batch_payload = []
    for item in b_items:
        txt_p, ann_p, text, gt_ents, doc_id = item
        doc_data = train_candidates_cache[doc_id]
        cands = doc_data["candidates"]
        batch_payload.append({
            "abstract": text,
            "candidates": cands,
            "file_name": doc_id,
            "doc_id": doc_id,
            "gt_ents": gt_ents
        })

    # Execute all 5 abstracts concurrently in parallel worker threads
    batch_results = ace_pipeline.train_batch_parallel(
        batch_items=batch_payload,
        max_workers=MAX_WORKERS,
        save_outputs_dir=os.path.join(OUTPUT_DIR, "train_agent_logs")
    )

    # Log results for each abstract in the batch
    for r in batch_results:
        doc_id = r["doc_id"]
        pred = r["predicted"]
        refl = r["reflection"]
        cur = r["curator_output"]
        ops = r["operations"]
        used_b = r["bullet_ids_used"]
        key_ins = r["key_insight"]
        spans_n = r["spans_count"]
        esc_n = r.get("escalated_count", spans_n)
        saved_pct = (1.0 - (esc_n / max(1, spans_n))) * 100.0

        print(f"  📄 Abstract {doc_id} ({spans_n} cands, {esc_n} escalated [{saved_pct:.1f}% saved]):")
        print(f"     ⚡ Gen Classified: {len(pred.get('final_answer', []))} spans (Used bullets: {used_b or 'Base Rules'})")
        print(f"     🧠 Refl Insight  : {key_ins[:80] if key_ins else 'None'}...")
        if ops:
            for op_idx, op in enumerate(ops, 1):
                op_type = op.get("type", "ADD")
                sec = op.get("section", op.get("ground_truth_label", op.get("id", "")))
                rule_snip = (op.get("guideline") or op.get("content", ""))[:65]
                print(f"     🏛️ Curator [{op_idx}/{len(ops)}]: [{op_type}] -> {sec} | Rule: '{rule_snip}...'")
        else:
            print("     🛡️ Curator Action: [Safe-Fail / No Update]")

        train_logs.append(r)

    # Persist guidelines and save atomic checkpoint after each completed batch
    ace_pipeline.save_guidelines()
    try:
        with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "last_completed_batch": b_idx,
                "train_logs": train_logs,
                "processed_doc_ids": [r["doc_id"] for r in train_logs],
                "active_bullets": sum(len(b) for b in ace_pipeline.manager.get_dynamicGuidelines().values()) if isinstance(ace_pipeline.manager.get_dynamicGuidelines(), dict) else len(ace_pipeline.manager.get_dynamicGuidelines()),
                "timestamp": time.time()
            }, f, indent=2)
    except Exception as e:
        print(f"⚠️ Warning: Could not write checkpoint file ({e})")

    b_elapsed = time.time() - t_batch_start
    curr_bullets = sum(len(b) for b in ace_pipeline.manager.get_dynamicGuidelines().values()) if isinstance(ace_pipeline.manager.get_dynamicGuidelines(), dict) else len(ace_pipeline.manager.get_dynamicGuidelines())
    print(f"  ✅ [Batch {b_idx + 1} Done in {b_elapsed:.2f}s | Checkpoint Saved] Active Guidelines in Store: {curr_bullets} bullets.")

# Final save and summary
ace_pipeline.save_guidelines()

# Export full training logs
with open(FINAL_LOGS_PATH, "w", encoding="utf-8") as f:
    json.dump(train_logs, f, indent=2)

# Remove checkpoint file on successful completion of target batches
if os.path.exists(CHECKPOINT_PATH):
    try:
        os.remove(CHECKPOINT_PATH)
        print(f"🧹 Curation complete! Checkpoint cleaned up. Logs saved to: {FINAL_LOGS_PATH}")
    except Exception:
        pass

total_train_time = time.time() - step_start
updated_guidelines = ace_pipeline.manager.get_dynamicGuidelines()
final_bullet_count = sum(len(b) for b in updated_guidelines.values()) if isinstance(updated_guidelines, dict) else len(updated_guidelines)
init_count = initial_bullet_count if "initial_bullet_count" in globals() else 0

print()
print("=" * 96)
print(f"🏁 ACE Training Complete in {total_train_time:.2f}s ({total_train_time/max(1, len(curation_subset)):.2f}s / abstract)!")
print(f"📚 Dynamic Guidelines Playbook Evolved: {init_count} -> {final_bullet_count} active bullets.")
print("=" * 96)


🌐 Full Production Mode: Running all 240 abstracts.
🚀 Starting BERTized ACE Multi-Threaded Curation across 240 abstracts (B=5, Workers=5, Mode=per_span)...
✨ FORCE_RESTART active: Starting clean run from Batch 1 (ignoring previous checkpoints).

⚡ [Batch 1/48] Processing abstracts 1..5 in 5 parallel threads...
  📄 Abstract 10069129 (9 cands, 7 escalated [22.2% saved]):
     ⚡ Gen Classified: 7 spans (Used bullets: Base Rules)
     🧠 Refl Insight  : Tobacco-related behaviors and secondary smoke exposures fundamentally belong to ...
     🏛️ Curator [1/1]: [ADD] -> Substance_use | Rule: 'Distinguish tobacco-derived exposures from ambient environmental ...'
  📄 Abstract 10516985 (41 cands, 17 escalated [58.5% saved]):
     ⚡ Gen Classified: 17 spans (Used bullets: Base Rules)
     🧠 Refl Insight  : Organized sporting events, tournaments, and athletic contests refer to structura...
     🏛️ Curator [1/1]: [ADD] -> O | Rule: 'Organized sporting events, tournaments, leagues, and athletic con...

In [7]:
# ==============================================================================
# [CELL 7]: Export Curated Guidelines to Markdown & CSV & Snapshot Backup
# ==============================================================================
md_export_path = os.path.join(OUTPUT_DIR, "curated_dynamicGuidelines.md")
csv_export_path = os.path.join(OUTPUT_DIR, "curated_dynamicGuidelines.csv")

# 1. Export Markdown Guidelines
ace_pipeline.export_guidelines_markdown(md_export_path)
print(f"📄 Exported clean Markdown guidelines to: {md_export_path}")

# 2. Export Tabular CSV with Metrics
ace_pipeline.export_guidelines_csv(csv_export_path)
print(f"📊 Exported tabular CSV guidelines to   : {csv_export_path}")

# 3. CLI Management Stats
import os
os.system(f"python scripts/manage_guidelines.py --guidelines-path {DYNAMIC_GUIDELINES_PATH} --stats")


📄 Exported clean Markdown guidelines to: /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/curated_dynamicGuidelines.md
📊 Exported tabular CSV guidelines to   : /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/curated_dynamicGuidelines.csv


sh: python: command not found


32512

In [13]:
# ==============================================================================
# [ABLATION CELL 1]: Empty Guidelines Baseline (0 Dynamic Guidelines)
# ==============================================================================
# Clears the active Dynamic Guidebook so Generator operates with zero guidelines.
# After running this cell, run [CELL 8] -> [CELL 10] -> [CELL 11] below.
# ==============================================================================
import json
import os

print("=" * 80)
print("🧪 ABLATION SETUP: Empty Dynamic Guidebook Baseline (0 guidelines)")
print("=" * 80)

# 1. Update in-memory GuidebookStore to empty
ace_pipeline.manager.store.guidebook = {"guidebook": []}
ace_pipeline.manager.store._rebuild_index()
ace_pipeline.manager.store.save(backup=False)

# Optional: keep ChromaDB vector store in sync if initialized
if hasattr(ace_pipeline.manager, "vectors") and ace_pipeline.manager.vectors:
    try:
        ace_pipeline.manager.vectors.sync_with_store(ace_pipeline.manager.store)
    except Exception:
        pass

# 2. Verify Generator view is empty
test_gen_md = ace_pipeline.manager.get_dynamicGuidelines_for_generator_md()
active_count = len(list(ace_pipeline.manager.store.iter_guidelines()))

print(f"✅ Dynamic Guidebook emptied successfully!")
print(f"   • Active guidelines in store : {active_count}")
print(f"   • Generator guidelines view  : {test_gen_md}")
print(f"   • Disk path synced           : {ace_pipeline.manager.store.guidebook_file_path}")
print("=" * 80)
print("👉 NEXT STEP: Run [CELL 8], then [CELL 10], then [CELL 11] below to test & evaluate.")
print("=" * 80)


🧪 ABLATION SETUP: Empty Dynamic Guidebook Baseline (0 guidelines)
Removed 71 stale entries from ChromaDB.
✅ Dynamic Guidebook emptied successfully!
   • Active guidelines in store : 0
   • Generator guidelines view  : []
   • Disk path synced           : /Users/hmd/Documents/Workspace/Unified_Test/data/guidelines/dynamicGuidebook.json
👉 NEXT STEP: Run [CELL 8], then [CELL 10], then [CELL 11] below to test & evaluate.


In [19]:
# ==============================================================================
# [ABLATION CELL 2]: Filter Harmful Guidelines (Prune harmful > helpful)
# ==============================================================================
# Loads run_20260920_175035_bertized_ace_16k guidebook, removes all guidelines
# where harmful > helpful, and loads the remaining 59 guidelines into the pipeline.
# After running this cell, run [CELL 8] -> [CELL 10] -> [CELL 11] below.
# ==============================================================================
import json
import os

SOURCE_GUIDEBOOK_PATH = os.path.join(
    PROJECT_ROOT, 
    "output/bertized_ace/runs/run_20260920_175035_bertized_ace_16k/dynamicGuidebook.json"
)

print("=" * 80)
print("🧪 ABLATION SETUP: Pruning Harmful Guidelines (harmful > helpful)")
print(f"   Source: {SOURCE_GUIDEBOOK_PATH}")
print("=" * 80)

if not os.path.exists(SOURCE_GUIDEBOOK_PATH):
    raise FileNotFoundError(f"Source guidebook not found at: {SOURCE_GUIDEBOOK_PATH}")

with open(SOURCE_GUIDEBOOK_PATH, "r", encoding="utf-8") as f:
    source_data = json.load(f)

raw_guidelines = source_data.get("guidebook", source_data.get("guidelines", []))

retained_guidelines = []
pruned_guidelines = []

for g in raw_guidelines:
    metrics = g.get("usage_metrics", {})
    helpful = metrics.get("helpful", 0)
    harmful = metrics.get("harmful", 0)
    
    if harmful > helpful:
        pruned_guidelines.append({
            "id": g.get("id"),
            "helpful": helpful,
            "harmful": harmful,
            "usage": metrics.get("usage_count", 0),
            "span": g.get("span", ""),
            "gt": g.get("ground_truth_label", "")
        })
    else:
        retained_guidelines.append(g)

# 1. Update in-memory GuidebookStore with filtered guidelines
ace_pipeline.manager.store.guidebook = {"guidebook": retained_guidelines}
ace_pipeline.manager.store._rebuild_index()
ace_pipeline.manager.store.save(backup=False)

# Optional: keep ChromaDB vector store in sync if initialized
if hasattr(ace_pipeline.manager, "vectors") and ace_pipeline.manager.vectors:
    try:
        ace_pipeline.manager.vectors.sync_with_store(ace_pipeline.manager.store)
    except Exception:
        pass

# 2. Print pruning summary
print(f"📊 Filtering Summary:")
print(f"   • Total Source Guidelines   : {len(raw_guidelines)}")
print(f"   • Pruned Harmful Guidelines : {len(pruned_guidelines)}")
print(f"   • Retained Active Guidelines: {len(retained_guidelines)}")
print()
print("✂️ Pruned Guidelines List (harmful > helpful):")
for p in pruned_guidelines:
    print(f"   - {p['id']}: harmful={p['harmful']} vs helpful={p['helpful']} (usage={p['usage']}) | span: '{p['span']}' [GT: {p['gt']}]")

print()
print(f"✅ Active store updated & synced with {len(retained_guidelines)} guidelines!")
print("=" * 80)
print("👉 NEXT STEP: Run [CELL 8], then [CELL 10], then [CELL 11] below to test & evaluate.")
print("=" * 80)


🧪 ABLATION SETUP: Pruning Harmful Guidelines (harmful > helpful)
   Source: /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/runs/run_20260920_175035_bertized_ace_16k/dynamicGuidebook.json
Added 59 missing embeddings to ChromaDB.
📊 Filtering Summary:
   • Total Source Guidelines   : 71
   • Pruned Harmful Guidelines : 12
   • Retained Active Guidelines: 59

✂️ Pruned Guidelines List (harmful > helpful):
   - DG-0003: harmful=3 vs helpful=2 (usage=5) | span: 'endurance competitions' [GT: O]
   - DG-0008: harmful=3 vs helpful=0 (usage=3) | span: 'exhaust fan' [GT: O]
   - DG-0012: harmful=1 vs helpful=0 (usage=1) | span: 'virus infection through water contamination' [GT: Environmental_exposures]
   - DG-0015: harmful=7 vs helpful=6 (usage=13) | span: 'gasoline' [GT: O]
   - DG-0017: harmful=10 vs helpful=0 (usage=10) | span: 'child abuse' [GT: Environmental_exposures]
   - DG-0019: harmful=5 vs helpful=3 (usage=8) | span: 'region of residence' [GT: O]
   - DG-0028: harmful

In [20]:
# ==============================================================================
# [CELL 8]: Phase 3 - Testing & Inference on Unseen Abstracts (Multi-Threaded, B=5 on 400/val)
# ==============================================================================
# Full test evaluation by default across all 80 abstracts of 400/val:
TEST_EVAL_LIMIT = int(os.getenv("TEST_LIMIT", "0")) or len(test_files)
test_subset = test_files[:TEST_EVAL_LIMIT]

print("=" * 96)
print(f"🔍 Running Generator Inference with Curated Guidelines on {len(test_subset)} test abstracts (B={BATCH_SIZE}, Workers={MAX_WORKERS})...")
print("=" * 96)

all_test_preds = []
all_test_gts = [item[3] for item in test_subset]
spanner_baseline_test_preds = []

t0 = time.time()
num_test_batches = (len(test_subset) + BATCH_SIZE - 1) // BATCH_SIZE

for b_idx in range(num_test_batches):
    b_start = b_idx * BATCH_SIZE
    b_end = min(b_start + BATCH_SIZE, len(test_subset))
    b_items = test_subset[b_start:b_end]

    batch_payload = []
    for item in b_items:
        txt_p, ann_p, text, gt_ents, doc_id = item
        doc_data = test_candidates_cache[doc_id]
        cands = doc_data["candidates"]
        batch_payload.append({
            "abstract": text,
            "candidates": cands,
            "file_name": doc_id,
            "doc_id": doc_id
        })

        # Collect SpanNER proposer baseline predictions for direct comparison
        doc_spanner = []
        for c in cands:
            s_lbl = c["predicted_label"]
            if s_lbl and s_lbl.lower() != "o":
                doc_spanner.append({
                    "span_text": c["span_text"],
                    "start_char": c["start_char"],
                    "end_char": c["end_char"],
                    "label": s_lbl
                })
        spanner_baseline_test_preds.append(doc_spanner)

    # Run ACE Generator inference in parallel threads
    batch_preds = ace_pipeline.predict_batch_parallel(
        batch_items=batch_payload,
        max_workers=MAX_WORKERS,
        save_outputs_dir=os.path.join(OUTPUT_DIR, "test_agent_logs")
    )

    for i, preds in enumerate(batch_preds):
        all_test_preds.append(preds)
        doc_id = batch_payload[i]["doc_id"]
        cands_n = len(batch_payload[i]["candidates"])
        num_pos = sum(1 for p in preds if p["label"] != "O")
        num_pruned = len(preds) - num_pos
        global_idx = b_start + i + 1
        num_esc = sum(1 for p in preds if p.get("escalated_to_llm", True))
        num_conf = cands_n - num_esc
        num_vetoed = sum(1 for p in preds if "Margin Veto" in p.get("source", ""))
        veto_str = f" | {num_vetoed} margin vetoes" if num_vetoed > 0 else ""
        print(f"  • Abstract [{global_idx}/{len(test_subset)}]: {doc_id} -> {cands_n} candidates ({num_esc} escalated, {num_conf} confident{veto_str} | {num_pos} accepted, {num_pruned} rejected to 'O')")

test_elapsed = time.time() - t0
print()
print(f"✅ Inference complete in {test_elapsed:.2f}s ({test_elapsed/max(1, len(test_subset)):.2f}s / abstract across {len(test_subset)} abstracts).")


🔍 Running Generator Inference with Curated Guidelines on 80 test abstracts (B=5, Workers=5)...
  • Abstract [1/80]: 10398936 -> 16 candidates (9 escalated, 7 confident | 1 margin vetoes | 16 accepted, 0 rejected to 'O')
  • Abstract [2/80]: 10773774 -> 7 candidates (0 escalated, 7 confident | 7 accepted, 0 rejected to 'O')
  • Abstract [3/80]: 12055820 -> 39 candidates (8 escalated, 31 confident | 39 accepted, 0 rejected to 'O')
  • Abstract [4/80]: 12799487 -> 16 candidates (3 escalated, 13 confident | 2 margin vetoes | 16 accepted, 0 rejected to 'O')
  • Abstract [5/80]: 12936919 -> 33 candidates (1 escalated, 32 confident | 1 margin vetoes | 33 accepted, 0 rejected to 'O')
  • Abstract [6/80]: 14740505 -> 8 candidates (3 escalated, 5 confident | 8 accepted, 0 rejected to 'O')
  • Abstract [7/80]: 15101565 -> 18 candidates (7 escalated, 11 confident | 4 margin vetoes | 18 accepted, 0 rejected to 'O')
  • Abstract [8/80]: 15572544 -> 29 candidates (7 escalated, 22 confident | 4 margin

## 🏆 Standardized Multi-Tier Evaluation Framework: Dual-View Benchmark

This evaluation framework decouples the performance of the **two architectural stages**:

### 1. View 1: Candidate-Level Arbitration Benchmark (Master Report / Sweep Engine Formulation)
- **Evaluates**: The decision accuracy of the ACE Generator on candidate spans proposed by SpanNER.
- **Recall Denominator**: Evaluated candidate spans matching ground truth (`TP / (TP + FN)` across candidate mentions).
- **Baseline Comparison**: Directly measures whether ACE Curation improved SpanNER's proposed candidate pool.

### 2. View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
- **Evaluates**: Total ground truth entity recovery against all gold entities in the test set across standard academic formulations:
  - **Strict CoNLL (1)**: Exact character boundary + exact category match.
  - **Partial MUC-7 (3)**: Any character boundary overlap + exact category match (1-to-1 bipartite lockout).
  - **Candidate Mention (4)**: Candidate mention coverage without 1-to-1 lockout against all gold entities.


In [21]:
# ==============================================================================
# [CELL 10]: Phase 4 - Standardized Dual-View Evaluation Benchmark
# View 1: Candidate Arbitration Benchmark (Decision Accuracy on Proposed Mentions)
# View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
# ==============================================================================
# ------------------------------------------------------------------------------
# Build Flat Audit Records across all test abstracts
# ------------------------------------------------------------------------------
all_audit_records = []
all_outcomes = []

for doc_idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(test_subset):
    preds = all_test_preds[doc_idx]
    for p in preds:
        gt_lbl = match_gt_label(p["span_text"], p["start_char"], p["end_char"], gt_ents)
        s_lbl = canonicalize_label(p.get("spanner_label", "O"), use_new_labels=USE_NEW_LABELS)
        f_lbl = canonicalize_label(p.get("label", "O"), use_new_labels=USE_NEW_LABELS)
        
        all_audit_records.append({
            "span": p["span_text"],
            "spanner_label": s_lbl,
            "hybrid_label": f_lbl,
            "ground_truth": gt_lbl
        })
        outcome_str = classify_10_outcome_str(gt_label=gt_lbl, spanner_label=s_lbl, final_label=f_lbl, escalated=True)
        all_outcomes.append(outcome_str)

# ------------------------------------------------------------------------------
# 1. VIEW 1: Candidate-Level Arbitration Benchmark
# ------------------------------------------------------------------------------
classes = sorted(list(set(r["ground_truth"] for r in all_audit_records if r["ground_truth"] != "O") |
                      set(r["hybrid_label"] for r in all_audit_records if r["hybrid_label"] != "O") |
                      set(r["spanner_label"] for r in all_audit_records if r["spanner_label"] != "O")))

per_class_table = []
tot_s_tp = tot_s_fp = tot_s_fn = 0
tot_h_tp = tot_h_fp = tot_h_fn = 0
s_f1_list, h_f1_list = [], []

for cls in classes:
    s_tp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] == cls)
    s_fp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] != cls)
    s_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["spanner_label"] != cls)
    
    h_tp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] == cls)
    h_fp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] != cls)
    h_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["hybrid_label"] != cls)
    
    s_p = s_tp / (s_tp + s_fp) if (s_tp + s_fp) > 0 else 0.0
    s_r = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0.0
    s_f1 = (2 * s_p * s_r) / (s_p + s_r) if (s_p + s_r) > 0 else 0.0
    
    h_p = h_tp / (h_tp + h_fp) if (h_tp + h_fp) > 0 else 0.0
    h_r = h_tp / (h_tp + h_fn) if (h_tp + h_fn) > 0 else 0.0
    h_f1 = (2 * h_p * h_r) / (h_p + h_r) if (h_p + h_r) > 0 else 0.0
    
    tot_s_tp += s_tp; tot_s_fp += s_fp; tot_s_fn += s_fn
    tot_h_tp += h_tp; tot_h_fp += h_fp; tot_h_fn += h_fn
    
    s_f1_list.append(s_f1); h_f1_list.append(h_f1)
    delta = h_f1 - s_f1
    delta_str = f"+{delta*100:.2f}%" if delta >= 0 else f"{delta*100:.2f}%"
    per_class_table.append((cls, s_f1, h_f1, delta_str, h_tp, h_fp, h_fn))

s_mic_p = tot_s_tp / (tot_s_tp + tot_s_fp) if (tot_s_tp + tot_s_fp) > 0 else 0.0
s_mic_r = tot_s_tp / (tot_s_tp + tot_s_fn) if (tot_s_tp + tot_s_fn) > 0 else 0.0
s_mic_f1 = (2 * s_mic_p * s_mic_r) / (s_mic_p + s_mic_r) if (s_mic_p + s_mic_r) > 0 else 0.0
s_mac_f1 = float(np.mean(s_f1_list)) if s_f1_list else 0.0

h_mic_p = tot_h_tp / (tot_h_tp + tot_h_fp) if (tot_h_tp + tot_h_fp) > 0 else 0.0
h_mic_r = tot_h_tp / (tot_h_tp + tot_h_fn) if (tot_h_tp + tot_h_fn) > 0 else 0.0
h_mic_f1 = (2 * h_mic_p * h_mic_r) / (h_mic_p + h_mic_r) if (h_mic_p + h_mic_r) > 0 else 0.0
h_mac_f1 = float(np.mean(h_f1_list)) if h_f1_list else 0.0

print("=" * 114)
print("🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Decision Accuracy on Proposed Mentions)")
print("   Evaluates decision accuracy on candidate mentions (SpanNER Proposer -> ACE Dynamic Guideline Arbitration)")
print("=" * 114)
print(f"{'Class Name':<48} | {'SpanNER F1':<10} | {'ACE Gen F1':<10} | {'Delta':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5}")
print("-" * 114)
for cls, s_f1, h_f1, d_str, tp, fp, fn in per_class_table:
    print(f"{cls:<48} | {s_f1*100:8.2f}% | {h_f1*100:8.2f}% | {d_str:<8} | {tp:<5} | {fp:<5} | {fn:<5}")
print("-" * 114)
print(f"Candidate Micro Precision : {h_mic_p*100:.2f}%  (SpanNER: {s_mic_p*100:.2f}%, Gain: +{(h_mic_p-s_mic_p)*100:.2f}%)")
print(f"Candidate Micro Recall    : {h_mic_r*100:.2f}%  (SpanNER: {s_mic_r*100:.2f}%, Gain: +{(h_mic_r-s_mic_r)*100:.2f}%)")
print(f"Candidate Micro F1-Score  : {h_mic_f1*100:.2f}%  (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)")
print(f"Macro F1-Score (Per-Class): {h_mac_f1*100:.2f}%  (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)")
print("=" * 114)

# ------------------------------------------------------------------------------
# 2. VIEW 2: Global End-to-End Dataset Benchmark (Academic Formulations)
# ------------------------------------------------------------------------------
spanner_m = [evaluate_doc_4_formulations(spanner_baseline_test_preds[i], all_test_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(test_subset))]
hybrid_m = [evaluate_doc_4_formulations(all_test_preds[i], all_test_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(test_subset))]

spanner_results = aggregate_4_formulations(spanner_m)
hybrid_results = aggregate_4_formulations(hybrid_m)

summary_box = format_4_formulations_summary_box(
    spanner_results=spanner_results,
    hybrid_results=hybrid_results,
    active_metrics=["strict", "partial", "mention"],
    system_name="SpanNER + ACE Curated Generator"
)
print("\n" + summary_box)

total_gold_entities = sum(len(gts) for gts in all_test_gts)
covered_gold_entities = hybrid_results["mention"]["tp"]
missed_proposals = total_gold_entities - covered_gold_entities

print("\n" + "=" * 114)
print("💡 ARCHITECTURAL PERFORMANCE ANALYSIS:")
print(f"• Phase 2 (Candidate Arbitration) : {h_mic_f1*100:.2f}% Micro F1 on proposed candidates.")
print(f"• Phase 1 (Span Proposal Ceiling) : Evaluated across all {total_gold_entities} gold entities in test set.")
print(f"  Base SpanNER proposed spans covering {covered_gold_entities} entities, leaving {missed_proposals} entities unproposed.")
print(f"  This retrieval ceiling produces a Global Recall of {hybrid_results['mention']['r']*100:.2f}% and Global Micro F1 of {hybrid_results['mention']['f1']*100:.2f}%.")
print("=" * 114)


🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Decision Accuracy on Proposed Mentions)
   Evaluates decision accuracy on candidate mentions (SpanNER Proposer -> ACE Dynamic Guideline Arbitration)
Class Name                                       | SpanNER F1 | ACE Gen F1 | Delta    | TP    | FP    | FN   
------------------------------------------------------------------------------------------------------------------
Environmental_exposures                          |    85.23% |    87.29% | +2.05%   | 127   | 30    | 7    
Mental_health_practices                          |    85.42% |    85.42% | +0.00%   | 41    | 1     | 13   
Non-LSF                                          |     0.00% |    10.78% | +10.78%  | 11    | 57    | 125  
Non_physical_leisure_time_activities             |    73.33% |    76.92% | +3.59%   | 45    | 23    | 4    
Nutrition                                        |    79.11% |    81.82% | +2.71%   | 333   | 138   | 10   
Personal_care_products_and_cosmetic_p

In [22]:
# ==============================================================================
# [CELL 11]: Phase 5 - Export Full Run Reports, JSON Metrics & Abstract Logs
# ==============================================================================
import json
import os
import re
import shutil
from datetime import datetime
from src.common.audit_logger import (
    compute_canonical_audit_breakdown,
    format_canonical_audit_table,
    canonicalize_label,
    match_gt_label,
    classify_10_outcome_str,
    log_bertized_ace_abstract_audit
)

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder_name = f"run_{timestamp_str}_bertized_ace_16k"
base_export_dir = os.path.join(OUTPUT_DIR, "runs", run_folder_name)
abstracts_export_dir = os.path.join(base_export_dir, "abstracts")
test_abstracts_dir = os.path.join(abstracts_export_dir, "test")
train_abstracts_dir = os.path.join(abstracts_export_dir, "train")

os.makedirs(test_abstracts_dir, exist_ok=True)
os.makedirs(train_abstracts_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# 0. Save Dynamic Guidebook Snapshot into Run Folder
# ------------------------------------------------------------------------------
guidebook_snapshot_dest = os.path.join(base_export_dir, "dynamicGuidebook.json")
if os.path.exists(DYNAMIC_GUIDELINES_PATH):
    shutil.copy2(DYNAMIC_GUIDELINES_PATH, guidebook_snapshot_dest)
    print(f"  • Dynamic Guidebook: Saved snapshot to {guidebook_snapshot_dest}")
elif os.path.exists(os.path.join(OUTPUT_DIR, "dynamicGuidelines.json")):
    shutil.copy2(os.path.join(OUTPUT_DIR, "dynamicGuidelines.json"), guidebook_snapshot_dest)
    print(f"  • Dynamic Guidebook: Saved fallback snapshot to {guidebook_snapshot_dest}")

print("=" * 105)
print(f"  💾 EXPORTING BERTIZED ACE RUN REPORTS & ABSTRACT AUDIT LOGS")
print("=" * 105)
print(f"  • Output Directory : {base_export_dir}")
print(f"  • Total Abstracts  : {len(test_subset)}")

# ------------------------------------------------------------------------------
# 1. Save Per-Abstract TXT Audit Files & Accumulate Escalated Tranche Metrics
# ------------------------------------------------------------------------------
def is_pos(lbl):
    l = canonicalize_label(lbl, use_new_labels=USE_NEW_LABELS)
    return l not in ("O", "Non-LSF", "Non_LSF", "LSF_out_of_context", "None", "")

esc_s_tp = esc_s_fp = esc_s_fn = 0
esc_h_tp = esc_h_fp = esc_h_fn = 0
tot_esc_spans = 0
tot_unesc_spans = 0

pos_corr_cat = 0   # GT_X_BERT_Y_LLM_X (Category corrected to true LSF)
pos_corr_drop = 0  # GT_O_BERT_X_LLM_O (Spurious FP dropped to O)
harm_corrupt = 0   # GT_X_BERT_X_LLM_Y (Correct LSF corrupted to wrong LSF)
harm_drop = 0      # GT_X_BERT_X_LLM_O (Correct LSF wrongly dropped to O)
pres_correct = 0   # GT_X_BERT_X_LLM_X (Correct LSF retained)

test_logs_dir = os.path.join(OUTPUT_DIR, "test_agent_logs")

for doc_idx, (txt_p, ann_p, text, gt_ents, doc_id) in enumerate(test_subset):
    preds = all_test_preds[doc_idx]

    # Extract Generator prompt and response for this abstract
    gen_prompt = ""
    gen_raw = ""
    for p in preds:
        if p.get("generator_prompt"):
            gen_prompt = p["generator_prompt"]
            gen_raw = p.get("generator_raw_response", "")
            break
    if not gen_prompt and os.path.exists(test_logs_dir):
        p_path = os.path.join(test_logs_dir, f"{doc_id}_generator_prompt.txt")
        r_path = os.path.join(test_logs_dir, f"{doc_id}_generator.txt")
        if os.path.exists(p_path):
            with open(p_path, "r", encoding="utf-8") as f:
                gen_prompt = f.read()
        if os.path.exists(r_path):
            with open(r_path, "r", encoding="utf-8") as f:
                gen_raw = f.read()

    # Log test abstract audit file (Raw Text, Matrix, Generator - omitting inactive Reflector/Curator)
    log_bertized_ace_abstract_audit(
        doc_id=doc_id,
        text=text,
        gt_ents=gt_ents,
        preds=preds,
        abstracts_dir=test_abstracts_dir,
        timestamp_str=timestamp_str,
        generator_prompt=gen_prompt,
        generator_raw_response=gen_raw,
        is_test_mode=True,
        use_new_labels=USE_NEW_LABELS
    )

    for p in preds:
        st_c, end_c = p["start_char"], p["end_char"]
        gt_raw = match_gt_label(p["span_text"], st_c, end_c, gt_ents)
        gt_lbl = canonicalize_label(gt_raw, use_new_labels=USE_NEW_LABELS)
        s_lbl = canonicalize_label(p.get("spanner_label", "O"), use_new_labels=USE_NEW_LABELS)
        f_lbl = canonicalize_label(p.get("label", "O"), use_new_labels=USE_NEW_LABELS)
        is_esc = p.get("escalated_to_llm", False)

        # Tally metrics for the escalated subpopulation
        gt_p = is_pos(gt_lbl)
        s_p = is_pos(s_lbl)
        f_p = is_pos(f_lbl)

        if is_esc:
            tot_esc_spans += 1
            # SpanNER candidate stats
            if s_p:
                if gt_p and s_lbl == gt_lbl: esc_s_tp += 1
                else: esc_s_fp += 1
            if gt_p and (not s_p or s_lbl != gt_lbl):
                esc_s_fn += 1
                
            # ACE candidate stats
            if f_p:
                if gt_p and f_lbl == gt_lbl: esc_h_tp += 1
                else: esc_h_fp += 1
            if gt_p and (not f_p or f_lbl != gt_lbl):
                esc_h_fn += 1

            # Arbitration dynamics breakdown
            if gt_p:
                if s_p and s_lbl == gt_lbl:
                    if f_p and f_lbl == gt_lbl: pres_correct += 1
                    elif f_p and f_lbl != gt_lbl: harm_corrupt += 1
                    else: harm_drop += 1
                elif s_p and s_lbl != gt_lbl:
                    if f_p and f_lbl == gt_lbl: pos_corr_cat += 1
            else:
                if s_p and not f_p:
                    pos_corr_drop += 1
        else:
            tot_unesc_spans += 1

# ------------------------------------------------------------------------------
# 1b. Save Per-Abstract TXT Audit Files for Training Abstracts (3 Agents)
# ------------------------------------------------------------------------------
train_agent_logs_dir = os.path.join(OUTPUT_DIR, "train_agent_logs")
if 'train_logs' in locals() and train_logs:
    print(f"  • Exporting {len(train_logs)} training abstract audits (3 agents: Gen, Refl, Cur) to {train_abstracts_dir}...")
    for entry in train_logs:
        doc_id = entry.get("doc_id")
        if not doc_id:
            continue
        doc_data = train_candidates_cache.get(doc_id, {})
        t_text = doc_data.get("text", "")
        t_cands = doc_data.get("candidates", [])
        t_gt_ents = doc_data.get("gt_ents", [])

        # Generator raw response and prompt
        gen_raw_path = os.path.join(train_agent_logs_dir, f"{doc_id}_generator.txt")
        t_gen_raw = open(gen_raw_path, "r", encoding="utf-8").read() if os.path.exists(gen_raw_path) else entry.get("predicted", {}).get("raw_response", "")
        gen_prompt_path = os.path.join(train_agent_logs_dir, f"{doc_id}_generator_prompt.txt")
        t_gen_prompt = open(gen_prompt_path, "r", encoding="utf-8").read() if os.path.exists(gen_prompt_path) else entry.get("predicted", {}).get("prompt", "")

        # Clean double curlies in generator output
        cleaned_gen = re.sub(r"\{\{+", "{", t_gen_raw.strip())
        cleaned_gen = re.sub(r"\}\}+", "}", cleaned_gen)
        parsed_gen = {}
        try:
            parsed_gen = json.loads(cleaned_gen)
        except Exception:
            pass

        llm_answers = parsed_gen.get("llm_answer", entry.get("predicted", {}).get("llm_answer", []))
        ans_by_id = {a.get("id"): a for a in llm_answers if isinstance(a, dict) and "id" in a}
        ans_by_text = {a.get("entity", "").strip().lower(): a for a in llm_answers if isinstance(a, dict)}

        train_preds = []
        for idx, c in enumerate(t_cands, 1):
            cid = c.get("id", idx)
            st_t = c.get("span_text", c.get("entity", ""))
            s_lbl = canonicalize_label(c.get("bert_predicted_label", c.get("predicted_label", c.get("spanner_label", "O"))), use_new_labels=USE_NEW_LABELS)
            ans = ans_by_id.get(cid) or ans_by_text.get(st_t.strip().lower())
            if ans:
                f_lbl = canonicalize_label(ans.get("llm_predicted_label", ans.get("final_label", s_lbl)), use_new_labels=USE_NEW_LABELS)
                is_esc = True
                b_ids = ans.get("bullet_ids", [])
            else:
                f_lbl = s_lbl
                is_esc = False
                b_ids = []
            train_preds.append({
                "span_text": st_t,
                "start_char": c.get("start_char", 0),
                "end_char": c.get("end_char", 0),
                "label": f_lbl,
                "spanner_label": s_lbl,
                "escalated_to_llm": is_esc,
                "bullet_ids": b_ids
            })

        # Reflector prompt and response
        ref_p_path = os.path.join(train_agent_logs_dir, f"{doc_id}_reflector_prompt.txt")
        ref_r_path = os.path.join(train_agent_logs_dir, f"{doc_id}_reflector.txt")
        t_ref_prompt = open(ref_p_path, "r", encoding="utf-8").read() if os.path.exists(ref_p_path) else entry.get("reflection", {}).get("prompt", "")
        t_ref_raw = open(ref_r_path, "r", encoding="utf-8").read() if os.path.exists(ref_r_path) else entry.get("reflection", {}).get("raw_response", "")

        # Curator prompt and response
        cur_p_path = os.path.join(train_agent_logs_dir, f"{doc_id}_curator_prompt.txt")
        cur_r_path = os.path.join(train_agent_logs_dir, f"{doc_id}_curator.txt")
        t_cur_prompt = open(cur_p_path, "r", encoding="utf-8").read() if os.path.exists(cur_p_path) else entry.get("curator_output", {}).get("prompt", "")
        t_cur_raw = open(cur_r_path, "r", encoding="utf-8").read() if os.path.exists(cur_r_path) else entry.get("curator_output", {}).get("raw_response", "")

        log_bertized_ace_abstract_audit(
            doc_id=doc_id,
            text=t_text,
            gt_ents=t_gt_ents,
            preds=train_preds,
            abstracts_dir=train_abstracts_dir,
            timestamp_str=timestamp_str,
            generator_prompt=t_gen_prompt,
            generator_raw_response=t_gen_raw,
            reflector_prompt=t_ref_prompt,
            reflector_raw_response=t_ref_raw,
            curator_prompt=t_cur_prompt,
            curator_raw_response=t_cur_raw,
            is_test_mode=False,
            use_new_labels=USE_NEW_LABELS
        )

# ------------------------------------------------------------------------------
# 2. Compute Escalated Benchmark & Arbitration Balance Sheet
# ------------------------------------------------------------------------------
tot_spans = tot_esc_spans + tot_unesc_spans
esc_rate = (tot_esc_spans / max(1, tot_spans)) * 100.0

esc_s_prec = esc_s_tp / max(1, esc_s_tp + esc_s_fp)
esc_s_rec  = esc_s_tp / max(1, esc_s_tp + esc_s_fn)
esc_s_f1   = (2 * esc_s_prec * esc_s_rec) / max(1e-9, esc_s_prec + esc_s_rec)

esc_h_prec = esc_h_tp / max(1, esc_h_tp + esc_h_fp)
esc_h_rec  = esc_h_tp / max(1, esc_h_tp + esc_h_fn)
esc_h_f1   = (2 * esc_h_prec * esc_h_rec) / max(1e-9, esc_h_prec + esc_h_rec)

tot_pos_corr = pos_corr_cat + pos_corr_drop
tot_harm_inv = harm_corrupt + harm_drop
net_corrections = tot_pos_corr - tot_harm_inv
win_rate = (tot_pos_corr / max(1, tot_pos_corr + tot_harm_inv)) * 100.0
acc_swing = (net_corrections / max(1, tot_esc_spans)) * 100.0

# Canonical 10-Outcome Diagnostic Breakdown
audit_breakdown = compute_canonical_audit_breakdown(all_test_preds, all_test_gts, use_new_labels=USE_NEW_LABELS)
audit_table_str = format_canonical_audit_table(audit_breakdown, method="ACE (GATED)", use_new_labels=USE_NEW_LABELS)

# ------------------------------------------------------------------------------
# 3. Save run_summary_metrics.json (Enriched with Escalated Dynamics)
# ------------------------------------------------------------------------------
metrics_json = {
    "architecture": "BERTized_ACE_Pipeline",
    "generator_prompt": "ACE_Generator_v3_bob.txt",
    "reflector_prompt": "ACE_Reflector_v3_bob.txt",
    "curator_prompt": "ACE_Curator_v3_per_span.txt" if CURATOR_MODE == "per_span" else "ACE_Curator_v3_bob.txt",
    "curator_mode": CURATOR_MODE,
    "timestamp": timestamp_str,
    "model_name": MODEL_NAME,
    "backend": BACKEND,
    "provider": PROVIDER,
    "max_tokens": MAX_TOKENS,
    "total_test_abstracts": len(test_subset),
    "candidate_arbitration_all_spans": {
        "spanner_micro_f1": s_mic_f1,
        "ace_micro_f1": h_mic_f1,
        "delta": h_mic_f1 - s_mic_f1,
        "spanner_macro_f1": s_mac_f1,
        "ace_macro_f1": h_mac_f1
    },
    "metrics_3_formulations_global": {
        "strict_conll": {"spanner": spanner_results["strict"], "ace": hybrid_results["strict"]},
        "partial_muc7": {"spanner": spanner_results["partial"], "ace": hybrid_results["partial"]},
        "candidate_mention": {"spanner": spanner_results["mention"], "ace": hybrid_results["mention"]}
    },
    "escalated_subpopulation_benchmark": {
        "total_spans": tot_spans,
        "escalated_spans": tot_esc_spans,
        "unescalated_spans": tot_unesc_spans,
        "escalation_rate_pct": esc_rate,
        "spanner_on_escalated": {"tp": esc_s_tp, "fp": esc_s_fp, "fn": esc_s_fn, "p": esc_s_prec, "r": esc_s_rec, "f1": esc_s_f1},
        "ace_on_escalated": {"tp": esc_h_tp, "fp": esc_h_fp, "fn": esc_h_fn, "p": esc_h_prec, "r": esc_h_rec, "f1": esc_h_f1},
        "delta_on_escalated": {
            "delta_tp": esc_h_tp - esc_s_tp,
            "delta_fp": esc_h_fp - esc_s_fp,
            "delta_p": esc_h_prec - esc_s_prec,
            "delta_r": esc_h_rec - esc_s_rec,
            "delta_f1": esc_h_f1 - esc_s_f1
        }
    },
    "arbitration_dynamics": {
        "positive_corrections": tot_pos_corr,
        "category_fixes": pos_corr_cat,
        "spurious_fp_pruned": pos_corr_drop,
        "harmful_inversions": tot_harm_inv,
        "category_corruptions": harm_corrupt,
        "false_rejections": harm_drop,
        "preserved_correct": pres_correct,
        "net_corrected_spans": net_corrections,
        "arbitration_win_rate_pct": win_rate,
        "accuracy_swing_pct": acc_swing
    },
    "canonical_10_outcomes": audit_breakdown
}

with open(os.path.join(base_export_dir, "run_summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, indent=4)

# ------------------------------------------------------------------------------
# 4. Save & Display run_summary_report.txt
# ------------------------------------------------------------------------------
report_lines = [
    "=" * 105,
    "📊 BERTIZED ACE EXPERIMENT RUN SUMMARY REPORT",
    f"Timestamp: {timestamp_str} | Total Test Abstracts: {len(test_subset)} | Mode: {CURATOR_MODE.upper()}",
    f"Provider : {PROVIDER} ({API_BASE_URL}) | Model: {MODEL_NAME} | Max Tokens: {MAX_TOKENS}",
    "=" * 105,
    "",
    "1. CANDIDATE ARBITRATION BENCHMARK (All Proposed Mentions)",
    "-" * 105,
    f"  • Candidate Micro F1-Score : {h_mic_p*100:.2f}% / {h_mic_r*100:.2f}% / {h_mic_f1*100:.2f}% (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)",
    f"  • Macro F1-Score (Classes) : {h_mac_f1*100:.2f}% (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)",
    "",
    "2. GLOBAL END-TO-END DATASET BENCHMARK (All Gold Entities)",
    "-" * 105,
    f"{'Formulation':<25} | {'SpanNER Baseline (P/R/F1)':<28} | {'SpanNER + ACE Gen (P/R/F1)':<28} | Gain / Delta",
    "-" * 105,
    f"{'Strict CoNLL (1)':<25} | {spanner_results['strict']['p']*100:>5.1f}% / {spanner_results['strict']['r']*100:>5.1f}% / {spanner_results['strict']['f1']*100:>5.1f}% | {hybrid_results['strict']['p']*100:>5.1f}% / {hybrid_results['strict']['r']*100:>5.1f}% / {hybrid_results['strict']['f1']*100:>5.1f}% | {(hybrid_results['strict']['f1'] - spanner_results['strict']['f1'])*100:>+6.2f}% F1",
    f"{'Partial MUC-7 (3)':<25} | {spanner_results['partial']['p']*100:>5.1f}% / {spanner_results['partial']['r']*100:>5.1f}% / {spanner_results['partial']['f1']*100:>5.1f}% | {hybrid_results['partial']['p']*100:>5.1f}% / {hybrid_results['partial']['r']*100:>5.1f}% / {hybrid_results['partial']['f1']*100:>5.1f}% | {(hybrid_results['partial']['f1'] - spanner_results['partial']['f1'])*100:>+6.2f}% F1",
    f"{'Candidate Mention (4)':<25} | {spanner_results['mention']['p']*100:>5.1f}% / {spanner_results['mention']['r']*100:>5.1f}% / {spanner_results['mention']['f1']*100:>5.1f}% | {hybrid_results['mention']['p']*100:>5.1f}% / {hybrid_results['mention']['r']*100:>5.1f}% / {hybrid_results['mention']['f1']*100:>5.1f}% | {(hybrid_results['mention']['f1'] - spanner_results['mention']['f1'])*100:>+6.2f}% F1",
    "",
    "3. ESCALATED SUBPOPULATION BENCHMARK (Local Lift on Ambiguous Spans)",
    "-" * 105,
    f"  • Escalated Spans Volume    : {tot_esc_spans} / {tot_spans} spans ({esc_rate:.1f}% escalated, {100-esc_rate:.1f}% gated locally)",
    f"  • SpanNER on Escalated      : Precision = {esc_s_prec*100:.2f}%, Recall = {esc_s_rec*100:.2f}%, Micro F1 = {esc_s_f1*100:.2f}% (FP = {esc_s_fp})",
    f"  • ACE Gen on Escalated      : Precision = {esc_h_prec*100:.2f}%, Recall = {esc_h_rec*100:.2f}%, Micro F1 = {esc_h_f1*100:.2f}% (FP = {esc_h_fp})",
    f"  • Isolated Escalated Gain   : Delta P = {(esc_h_prec-esc_s_prec)*100:+.2f}%, Delta R = {(esc_h_rec-esc_s_rec)*100:+.2f}%, Delta F1 = {(esc_h_f1-esc_s_f1)*100:+.2f}%",
    f"  • False Positive Reduction  : {esc_s_fp - esc_h_fp} spurious spans eliminated ({esc_s_fp} -> {esc_h_fp})",
    "",
    "4. ARBITRATION BALANCE SHEET & WIN RATE (Correction vs Degradation Dynamics)",
    "-" * 105,
    f"  • Beneficial Corrections (Positive) : {tot_pos_corr} spans (+{pos_corr_cat} category fixes, +{pos_corr_drop} false positives dropped to O)",
    f"  • Harmful Inversions (Degradations) : {tot_harm_inv} spans (-{harm_corrupt} category corruptions, -{harm_drop} false drops to O)",
    f"  • Preserved Correct Ground Truth    : {pres_correct} spans accurately validated and maintained",
    f"  • Net Corrected Spans               : {net_corrections:+d} spans ({acc_swing:+.2f}% net accuracy swing on escalated tranche)",
    f"  • Arbitration Win Rate              : {win_rate:.2f}% (Positive Corrections / Total Label Modifications)",
    "=" * 105,
    "",
    "-" * 105,
    "5. CANONICAL 10-OUTCOME DIAGNOSTIC BREAKDOWN",
    "-" * 105,
    audit_table_str,
    "=" * 105
]

with open(os.path.join(base_export_dir, "run_summary_report.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"✅ Successfully exported all run reports, metrics, and abstract logs to: {base_export_dir}")
print("\n" + "\n".join(report_lines))


  • Dynamic Guidebook: Saved snapshot to /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/runs/run_20260920_190129_bertized_ace_16k/dynamicGuidebook.json
  💾 EXPORTING BERTIZED ACE RUN REPORTS & ABSTRACT AUDIT LOGS
  • Output Directory : /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/runs/run_20260920_190129_bertized_ace_16k
  • Total Abstracts  : 80
  • Exporting 240 training abstract audits (3 agents: Gen, Refl, Cur) to /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/runs/run_20260920_190129_bertized_ace_16k/abstracts/train...
✅ Successfully exported all run reports, metrics, and abstract logs to: /Users/hmd/Documents/Workspace/Unified_Test/output/bertized_ace/runs/run_20260920_190129_bertized_ace_16k

📊 BERTIZED ACE EXPERIMENT RUN SUMMARY REPORT
Timestamp: 20260920_190129 | Total Test Abstracts: 80 | Mode: PER_SPAN
Provider : 9router (http://localhost:20128/v1) | Model: gemini-cli | Max Tokens: 16384

1. CANDIDATE ARBITRATION BENCHMARK

In [23]:
# ==============================================================================
# 🔍 INSPECT MULTI-INSIGHT PROPOSALS FROM REFLECTOR
# ==============================================================================
import json
import os

# Load logs from in-memory variable or disk
if "train_logs" in globals() and train_logs:
    logs = train_logs
else:
    log_path = os.path.join(OUTPUT_DIR, "train_logs.json")
    with open(log_path, "r", encoding="utf-8") as f:
        logs = json.load(f)

# Find all abstracts with > 1 insight
multi_insight_records = []
for entry in logs:
    doc_id = entry.get("doc_id")
    refl = entry.get("reflection", {})
    
    # Check 1: structured diagnostic_items list
    items = refl.get("diagnostic_items", [])
    insights = []
    if isinstance(items, list) and items:
        insights = [it.get("key_insight", "").strip() for it in items if isinstance(it, dict) and it.get("key_insight")]
    
    # Check 2: pipe-delimited fallback in key_insight string
    if not insights and isinstance(refl.get("key_insight"), str) and " | " in refl["key_insight"]:
        insights = [i.strip() for i in refl["key_insight"].split(" | ") if i.strip()]
        
    if len(insights) > 1:
        multi_insight_records.append({
            "doc_id": doc_id,
            "count": len(insights),
            "insights": insights,
            "items": items,
            "escalated_count": entry.get("escalated_count", 0),
            "operations": entry.get("operations", [])
        })

print("=" * 90)
print(f"📊 REFLECTOR MULTI-INSIGHT AUDIT SUMMARY")
print("=" * 90)
print(f"  • Total Training Abstracts Evaluated : {len(logs)}")
print(f"  • Abstracts with >1 Reflector Insight: {len(multi_insight_records)} ({len(multi_insight_records)/len(logs)*100:.1f}%)")
print("=" * 90)

# Display first 5 examples with detailed breakdown
for idx, rec in enumerate(multi_insight_records[:5], 1):
    print(f"\n[{idx}] 📄 DOC ID: {rec['doc_id']} | Proposed Insights: {rec['count']} | Escalated Spans: {rec['escalated_count']}")
    print("-" * 90)
    for i_num, item in enumerate(rec['items'] if rec['items'] else rec['insights'], 1):
        if isinstance(item, dict):
            outcome = item.get("primary_error_outcome", "N/A")
            insight_text = item.get("key_insight", "None")
            err_span = item.get("error_identification", "").split("\n")[0]
            print(f"   💡 Insight #{i_num} [{outcome}]:")
            print(f"      • Principle : {insight_text}")
            if err_span and err_span != "None":
                print(f"      • Span Info : {err_span}")
        else:
            print(f"   💡 Insight #{i_num}: {item}")
    
    ops = rec["operations"]
    print(f"   🏛️ Curator Operations Formulated: {len(ops)} (Curator Mode: {CURATOR_MODE if 'CURATOR_MODE' in globals() else 'N/A'})")


📊 REFLECTOR MULTI-INSIGHT AUDIT SUMMARY
  • Total Training Abstracts Evaluated : 240
  • Abstracts with >1 Reflector Insight: 134 (55.8%)

[1] 📄 DOC ID: 10069129 | Proposed Insights: 3 | Escalated Spans: 7
------------------------------------------------------------------------------------------
   💡 Insight #1 [N/A]:
      • Principle : Tobacco-related behaviors and secondary smoke exposures fundamentally belong to the substance use domain rather than general environmental contaminants. Evaluate the source of the exposure, distinguishing direct and indirect tobacco smoke interactions from non-substance environmental hazards.
      • Span Info : | passive smoking | GT_X_BERT_X_LLM_Y | [] | none | BERT proposed the correct LSF category (Substance_use), but the Generator misclassified it as Environmental_exposures. |
   💡 Insight #2 [N/A]:
      • Principle : Fixed candidate spans containing coordinated lifestyle concepts require identifying the primary behavioral domain rather than reje

In [24]:
# ==============================================================================
# 🏛️ AUDIT: CURATOR MULTIPLE GUIDELINE ADDITIONS & MODIFICATIONS
# ==============================================================================
import json
import os

# 1. Load logs from memory or checkpoint file
if "train_logs" in globals() and train_logs:
    logs = train_logs
else:
    log_path = os.path.join(OUTPUT_DIR, "train_logs.json")
    with open(log_path, "r", encoding="utf-8") as f:
        logs = json.load(f)

# 2. Categorize abstracts by Curator operations
multi_add_records = []
multi_ops_records = []
ops_distribution = {}

for entry in logs:
    doc_id = entry.get("doc_id")
    ops = entry.get("operations", []) or entry.get("curator_output", {}).get("operations", [])
    
    n_ops = len(ops)
    ops_distribution[n_ops] = ops_distribution.get(n_ops, 0) + 1
    
    add_ops = [op for op in ops if str(op.get("type", "")).upper() == "ADD"]
    mod_ops = [op for op in ops if str(op.get("type", "")).upper() == "MODIFY"]
    
    rec = {
        "doc_id": doc_id,
        "total_ops": n_ops,
        "adds": add_ops,
        "mods": mod_ops,
        "all_ops": ops,
        "insights_count": len(entry.get("reflection", {}).get("diagnostic_items", []))
    }
    
    if len(add_ops) > 1:
        multi_add_records.append(rec)
    if n_ops > 1:
        multi_ops_records.append(rec)

# 3. Print Aggregate Summary
print("=" * 105)
print("📊 CURATOR MULTI-OPERATION AUDIT REPORT")
print("=" * 105)
print(f"  • Total Abstracts Processed       : {len(logs)}")
print(f"  • Safe-Fail / 0 Operations (No-op): {ops_distribution.get(0, 0)} ({ops_distribution.get(0, 0)/len(logs)*100:.1f}%)")
print(f"  • Single Operation (1 rule update): {ops_distribution.get(1, 0)} ({ops_distribution.get(1, 0)/len(logs)*100:.1f}%)")
print(f"  • Multiple Operations (>1 update) : {len(multi_ops_records)} ({len(multi_ops_records)/len(logs)*100:.1f}%)")
print(f"    ├─ Multiple ADDs (>=2 New Rules): {len(multi_add_records)} abstracts")
print(f"    └─ Mixed / Multi-Modifications  : {len(multi_ops_records) - len(multi_add_records)} abstracts")
print("=" * 105)

# 4. Detailed Breakdown of Abstracts that Added MULTIPLE Brand-New Guidelines
print("\n" + "🔥" * 40)
print(f" ABSTRACTS THAT ADDED MULTIPLE NEW GUIDELINES (Total: {len(multi_add_records)})")
print("🔥" * 40)

for idx, rec in enumerate(multi_add_records, 1):
    doc_id = rec["doc_id"]
    adds = rec["adds"]
    mods = rec["mods"]
    print(f"\n[{idx}] 📄 DOC ID: {doc_id}  |  Reflector Insights: {rec['insights_count']}  |  Curator Added: {len(adds)} rules (+ {len(mods)} modified)")
    print("-" * 105)
    for a_idx, op in enumerate(adds, 1):
        target_cat = op.get("ground_truth_label") or op.get("section") or op.get("category", "O")
        bert_lbl = op.get("bert_label", "N/A")
        span = op.get("span", "N/A")
        rule_text = op.get("guideline") or op.get("content", "")
        triad = op.get("triad", {}).get("type", "N/A") if isinstance(op.get("triad"), dict) else "N/A"
        
        print(f"   ➕ [ADD #{a_idx}] Target Category: [{target_cat}] | Triad: {triad} | Trigger Span: '{span}' (BERT: {bert_lbl})")
        print(f"      📖 Curated Rule: {rule_text}")
    
    if mods:
        for m_idx, op in enumerate(mods, 1):
            target_id = op.get("id", "N/A")
            reason = op.get("reason", "Refined boundary")
            print(f"   🔄 [MODIFY #{m_idx}] Existing Rule ID: [{target_id}] | Rationale: {reason}")

# 5. Quick Sample of Mixed Operations (1 ADD + 1 MODIFY)
mixed_records = [r for r in multi_ops_records if len(r["adds"]) == 1 and len(r["mods"]) >= 1]
print("\n" + "=" * 105)
print(f"📋 SAMPLE ABSTRACTS WITH HYBRID OPERATIONS (1 ADD + 1 MODIFY) -  {len(mixed_records)}")
print("=" * 105)

for idx, rec in enumerate(mixed_records[:len(mixed_records)-1], 1):
    doc_id = rec["doc_id"]
    add_op = rec["adds"][0]
    mod_op = rec["mods"][0]
    print(f"\n• Doc {doc_id}:")
    print(f"  ➕ ADD    : Category [{add_op.get('ground_truth_label')}] on span '{add_op.get('span')}'")
    print(f"             Rule: {add_op.get('guideline')[:100]}...")
    print(f"  🔄 MODIFY : Rule [{mod_op.get('id')}] -> {mod_op.get('reason')}")


📊 CURATOR MULTI-OPERATION AUDIT REPORT
  • Total Abstracts Processed       : 240
  • Safe-Fail / 0 Operations (No-op): 130 (54.2%)
  • Single Operation (1 rule update): 84 (35.0%)
  • Multiple Operations (>1 update) : 26 (10.8%)
    ├─ Multiple ADDs (>=2 New Rules): 9 abstracts
    └─ Mixed / Multi-Modifications  : 17 abstracts

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
 ABSTRACTS THAT ADDED MULTIPLE NEW GUIDELINES (Total: 9)
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

[1] 📄 DOC ID: 14643283  |  Reflector Insights: 6  |  Curator Added: 2 rules (+ 1 modified)
---------------------------------------------------------------------------------------------------------
   ➕ [ADD #1] Target Category: [O] | Triad: GT_O_BERT_X_LLM_X | Trigger Span: 'exhaust fan' (BERT: Environmental_exposures)
      📖 Curated Rule: Standalone mechanical appliances, ventilation fixtures, and structural domestic hardware represent inert equipment rather than active lifestyle practices or ambient environmental hazards